<a href="https://colab.research.google.com/github/0002F16/hybrid-mobilenetv2-dualconv-eca/blob/main/colab/Eval_External_CIFAR10_Curated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Evaluate external curated CIFAR-10 dataset in Colab

This notebook is for **cloud evaluation** of the external ImageFolder-style CIFAR-10 dataset against your saved CIFAR-10 checkpoints.

It supports:
- **resize=32** full-dataset evaluation
- optional **resize=0** no-resize evaluation
- exporting `summary.json`, `report.md`, and zipped results back to Google Drive

### Required assets
Colab does not have your local files, so this notebook now defaults to:

1. **Dataset from GitHub Release asset**
   - Release page: `https://github.com/0002F16/hybrid-mobilenetv2-dualconv-eca/releases/tag/external-cifar10-curated-v1`
   - Asset: `cifar10_curated_dataset_for_colab.zip`
2. **Checkpoints from Google Drive**
   - recommended zip: `cifar10_checkpoints_for_colab.zip`
   - or extracted folder already in Drive

See `colab/PREP_EXTERNAL_CIFAR10_EVAL_FOR_COLAB.md` in the repo for prep commands.


In [ ]:
from pathlib import Path
import os

GIT_URL = "https://github.com/0002F16/hybrid-mobilenetv2-dualconv-eca"
REPO_DIR = Path("/content/hybrid-mobilenetv2-dualconv-eca")

if not REPO_DIR.exists():
    os.system(f"git clone {GIT_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print("Repo:", REPO_DIR)
os.system("pip -q install -r requirements.txt")


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

# ---- Change these if your Drive layout differs ----
DRIVE_ROOT = Path('/content/drive/MyDrive/hybrid-mobilenetv2-dualconv-eca')
DRIVE_INPUTS = DRIVE_ROOT / 'colab_inputs'
DRIVE_EXPORTS = DRIVE_ROOT / 'colab_exports'

CHECKPOINTS_ZIP = DRIVE_INPUTS / 'cifar10_checkpoints_for_colab.zip'

# Optional: use extracted checkpoints folder instead of zip
CHECKPOINTS_DIR_ON_DRIVE = DRIVE_INPUTS / 'Trained Models' / 'cifar10'

# Dataset defaults to GitHub Release asset (you can override with a Drive zip/folder if you want)
DATASET_RELEASE_URL = 'https://github.com/0002F16/hybrid-mobilenetv2-dualconv-eca/releases/download/external-cifar10-curated-v1/cifar10_curated_dataset_for_colab.zip'
DATASET_ZIP = DRIVE_INPUTS / 'cifar10_curated_dataset_for_colab.zip'
DATASET_DIR_ON_DRIVE = DRIVE_INPUTS / 'cifar10'

WORK_ROOT = Path('/content/external_cifar10_eval')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_EXPORTS.mkdir(parents=True, exist_ok=True)

print('DRIVE_INPUTS:', DRIVE_INPUTS)
print('DRIVE_EXPORTS:', DRIVE_EXPORTS)
print('DATASET_RELEASE_URL:', DATASET_RELEASE_URL)


In [ ]:
import shutil
import urllib.request
import zipfile
from pathlib import Path

def ensure_from_zip_or_dir(zip_path: Path, extracted_dir: Path | None, dest_dir: Path) -> Path:
    if extracted_dir is not None and extracted_dir.exists():
        print(f'Using extracted dir: {extracted_dir}')
        return extracted_dir
    if not zip_path.exists():
        raise FileNotFoundError(f'Missing zip file: {zip_path}')
    unzip_root = dest_dir
    unzip_root.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {zip_path} -> {unzip_root}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(unzip_root)
    return unzip_root

def ensure_dataset_source(zip_path: Path, extracted_dir: Path | None, dest_dir: Path, release_url: str | None) -> Path:
    if extracted_dir is not None and extracted_dir.exists():
        print(f'Using extracted dataset dir: {extracted_dir}')
        return extracted_dir
    local_zip = zip_path
    if not local_zip.exists():
        if not release_url:
            raise FileNotFoundError(f'Missing dataset zip file: {local_zip}')
        local_zip.parent.mkdir(parents=True, exist_ok=True)
        print(f'Downloading dataset release asset -> {local_zip}')
        urllib.request.urlretrieve(release_url, local_zip)
    unzip_root = dest_dir
    unzip_root.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {local_zip} -> {unzip_root}')
    with zipfile.ZipFile(local_zip, 'r') as zf:
        zf.extractall(unzip_root)
    return unzip_root

checkpoints_root_parent = WORK_ROOT / 'checkpoints_unzipped'
dataset_root_parent = WORK_ROOT / 'dataset_unzipped'

checkpoints_source = ensure_from_zip_or_dir(CHECKPOINTS_ZIP, CHECKPOINTS_DIR_ON_DRIVE if CHECKPOINTS_DIR_ON_DRIVE.exists() else None, checkpoints_root_parent)
dataset_source = ensure_dataset_source(DATASET_ZIP, DATASET_DIR_ON_DRIVE if DATASET_DIR_ON_DRIVE.exists() else None, dataset_root_parent, DATASET_RELEASE_URL)

# Normalize the expected paths regardless of zip layout
candidate_checkpoint_roots = [
    checkpoints_source,
    checkpoints_source / 'Trained Models' / 'cifar10',
    checkpoints_root_parent / 'Trained Models' / 'cifar10',
]
CHECKPOINTS_ROOT = next((p for p in candidate_checkpoint_roots if p.exists() and p.name == 'cifar10'), None)
if CHECKPOINTS_ROOT is None:
    matches = [p for p in checkpoints_root_parent.rglob('cifar10') if p.is_dir()]
    CHECKPOINTS_ROOT = matches[0] if matches else None
if CHECKPOINTS_ROOT is None:
    raise FileNotFoundError('Could not locate checkpoints root for cifar10 after extraction.')

candidate_dataset_roots = [
    dataset_source,
    dataset_source / 'cifar10',
    dataset_root_parent / 'cifar10',
]
DATASET_DIR = next((p for p in candidate_dataset_roots if p.exists() and p.name == 'cifar10'), None)
if DATASET_DIR is None:
    matches = [p for p in dataset_root_parent.rglob('cifar10') if p.is_dir()]
    DATASET_DIR = matches[0] if matches else None
if DATASET_DIR is None:
    raise FileNotFoundError('Could not locate dataset dir for cifar10 after extraction.')

print('CHECKPOINTS_ROOT =', CHECKPOINTS_ROOT)
print('DATASET_DIR =', DATASET_DIR)


In [ ]:
from pathlib import Path
import os

# Evaluation controls
RUN_RESIZE32 = True
RUN_NO_RESIZE = False  # set True if using a high-RAM Colab runtime and want to attempt it
RESIZE_VALUE = 32
DEVICE_RESIZE32 = 'auto'
DEVICE_NO_RESIZE = 'cpu'
BATCH_SIZE_RESIZE32 = 64
BATCH_SIZE_NO_RESIZE = 1
NUM_WORKERS = 2

# Optional filters
VARIANTS = []   # e.g. ['hybrid'] or [] for all
SEEDS = []      # e.g. [123] or [] for all

RESIZE32_OUT = REPO_DIR / 'outputs' / 'external_eval' / 'cifar10_curated_colab_resize32'
NO_RESIZE_OUT = REPO_DIR / 'outputs' / 'external_eval' / 'cifar10_curated_colab_no_resize'

print('RUN_RESIZE32:', RUN_RESIZE32)
print('RUN_NO_RESIZE:', RUN_NO_RESIZE)
print('VARIANTS:', VARIANTS or 'ALL')
print('SEEDS:', SEEDS or 'ALL')


In [ ]:
import subprocess
import sys

def run_eval(*, resize: int, device: str, batch_size: int, output_dir: Path):
    cmd = [
        sys.executable,
        'experiments/eval_cifar10_imagefolder_report.py',
        '--dataset_dir', str(DATASET_DIR),
        '--checkpoints_root', str(CHECKPOINTS_ROOT),
        '--official_data_root', str(REPO_DIR / 'data'),
        '--batch_size', str(batch_size),
        '--num_workers', str(NUM_WORKERS),
        '--device', device,
        '--resize', str(resize),
        '--output_dir', str(output_dir),
    ]
    if VARIANTS:
        cmd += ['--variants', *VARIANTS]
    if SEEDS:
        cmd += ['--seeds', *[str(s) for s in SEEDS]]

    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    print('RETURN CODE:', result.returncode)
    print('\n--- STDOUT ---\n')
    print(result.stdout)
    print('\n--- STDERR ---\n')
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError('eval script failed')


In [ ]:
if RUN_RESIZE32:
    run_eval(
        resize=RESIZE_VALUE,
        device=DEVICE_RESIZE32,
        batch_size=BATCH_SIZE_RESIZE32,
        output_dir=RESIZE32_OUT,
    )
else:
    print('Skipping resize=32 run')


In [ ]:
if RUN_NO_RESIZE:
    run_eval(
        resize=0,
        device=DEVICE_NO_RESIZE,
        batch_size=BATCH_SIZE_NO_RESIZE,
        output_dir=NO_RESIZE_OUT,
    )
else:
    print('Skipping no-resize run')


In [ ]:
from pathlib import Path

for out_dir in [RESIZE32_OUT, NO_RESIZE_OUT]:
    if out_dir.exists():
        print('
===', out_dir, '===')
        for name in ['report.md', 'summary.json']:
            p = out_dir / name
            print(name, 'exists:', p.exists())
            if p.exists() and name.endswith('.md'):
                print(p.read_text()[:2000])


In [ ]:
import shutil
from google.colab import files

EXPORT_BASE = DRIVE_EXPORTS / 'external_cifar10_eval'
EXPORT_BASE.mkdir(parents=True, exist_ok=True)

exported = []
for src in [RESIZE32_OUT, NO_RESIZE_OUT]:
    if src.exists():
        dst = EXPORT_BASE / src.name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        zip_path = shutil.make_archive(str(dst), 'zip', root_dir=dst)
        exported.append(Path(zip_path))
        print('Exported to Drive:', dst)
        print('Zipped:', zip_path)

print('Exported archives:')
for p in exported:
    print('-', p)

# Uncomment to download the first zip directly to your machine from Colab:
# if exported:
#     files.download(str(exported[0]))
